In [1]:
# ==================== VULNERABILITY COUNT MODELS ====================
library(fixest)
library(dplyr)
library(tidyr)
library(lubridate)

# ---- 1. Dichotomization functions -------------------------
bin_zero_pos <- function(x) {
  factor(if_else(x < 0.5, "zero", "positive"),
         levels = c("zero", "positive"))
}
bin_ten_less <- function(x) {
  factor(if_else(x > 9.5, "ten", "below_ten"),
         levels = c("below_ten", "ten"))
}
binnings <- list(zero_pos = bin_zero_pos, ten_less = bin_ten_less)

chosen_binning <- c(
  Maintained_score           = "continuous",
  Code_Review_score          = "continuous",
  Pinned_Dependencies_score  = "zero_pos",
  Security_Policy_score      = "zero_pos",
  Token_Permissions_score    = "zero_pos",
  DependUpTool_score         = "zero_pos",
  Binary_Artifacts_score     = "ten_less",
  Dangerous_Workflow_score   = "ten_less",
  Contrib_score              = "ten_less"
)

controls <- "log1p(downloads_7_day_total) + log1p(dependents) + log1p(loc) + version_age_days + log1p(Release_count)"
fe_part  <- "package_name + year_month"

# ---- 2. Function to run one univariate model -----------------------------
run_univariate_model <- function(panel, check_name, binning_type, outcome_var) {
  panel_check <- panel
  if (binning_type == "continuous") {
    predictor_term <- check_name
  } else {
    bin_col <- paste0(check_name, "_bin")
    f <- binnings[[binning_type]]
    panel_check[[bin_col]] <- f(panel_check[[check_name]])
    predictor_term <- bin_col
  }
  fml_str <- paste(
    outcome_var, "~", predictor_term, "+", controls,
    "|", fe_part
  )
  model <- feglm(as.formula(fml_str), data = panel_check,
                  family = "poisson", cluster = ~package_name)
  list(model = model, n_obs = nobs(model), check = check_name, outcome = outcome_var)
}

# ---- 3. Run for every check ------------------------------------------------
panel_vuln <- read.csv("../data/monthly_panel.csv")  # update path as needed

vuln_results <- lapply(names(chosen_binning), function(check_name) {
  run_univariate_model(panel_vuln, check_name, chosen_binning[[check_name]], "Vulnerability_count")
})
names(vuln_results) <- names(chosen_binning)

# ---- 4. Print summary() for each model -------------------------------------
cat("=== Vulnerability Count: Univariate Model Summaries ===\n\n")
for (check_name in names(vuln_results)) {
  cat("----", check_name, "----\n")
  print(summary(vuln_results[[check_name]]$model))
  cat("\n")
}

# ---- 5. N and % change summary per check -----------------------------------
summarize_univariate <- function(results_list, outcome_label) {
  for (check_name in names(results_list)) {
    res <- results_list[[check_name]]
    m <- res$model
    coef_name <- names(coef(m))[1]
    pct_change <- (exp(coef(m)[coef_name]) - 1) * 100
    cat(sprintf("[%s] %s (N=%d): coef on %s -> %.2f%% change\n",
                outcome_label, check_name, res$n_obs, coef_name, pct_change))
  }
}
summarize_univariate(vuln_results, "Vulnerability Count")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union


NOTES: 16,098 observations removed because of NA values (RHS: 16,098).
       5,183/0 fixed-effects (22,540 observations) removed because of only 0 outcomes or singletons.

NOTES: 16,130 observations removed because of NA values (RHS: 16,130).
       5,206/0 fixed-effects (22,664 observations) removed because of only 0 outcomes or singletons.

NOTES: 24,944 observations removed because of NA values (RHS: 24,944).
       4,489/0 fixed-effects (19,385 observations) removed because of only 0 outcomes or singletons.

NOTES: 15,965 observations removed because of NA values (RHS: 15,965).
       5,200/0 fixed-effects (22,673 observations) removed because of only 0 outco

=== Vulnerability Count: Univariate Model Summaries ===

---- Maintained_score ----
GLM estimation, family = poisson, Dep. Var.: Vulnerability_count
Observations: 65,416
Fixed-effects: package_name: 9,056,  year_month: 25
Standard-errors: Clustered (package_name) 
                              Estimate Std. Error  z value   Pr(>|z|)    
Maintained_score             -0.013989   0.002712 -5.15753 2.5023e-07 ***
log1p(downloads_7_day_total)  0.003315   0.002285  1.45036 1.4696e-01    
log1p(dependents)             0.011235   0.005459  2.05800 3.9591e-02 *  
log1p(loc)                    0.195186   0.028537  6.83977 7.9321e-12 ***
version_age_days             -0.001084   0.000318 -3.40414 6.6374e-04 ***
log1p(Release_count)         -0.018405   0.007533 -2.44323 1.4556e-02 *  
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -238,411.4   Adj. Pseudo R2: 0.810098
           BIC:  577,573.0     Squared Cor.: 0.884749

---- Code_Review_score ----
GLM estimatio

In [2]:
# ==================== TABLE III: INDIVIDUAL CHECK MODELS (LaTeX) ====================
library(fixest)

# ---- 1. Row labels + ordering (matches chosen_binning order) --------------
row_labels <- c(
  Maintained_score          = "Maintained",
  Code_Review_score         = "Code-Review",
  Pinned_Dependencies_score = "Pinned-Dependencies ($>0$)",
  Security_Policy_score     = "Security-Policy ($>0$)",
  Token_Permissions_score   = "Token-Permissions ($>0$)",
  DependUpTool_score        = "Dependency-Update-Tool ($>0$)",
  Binary_Artifacts_score    = "Binary-Artifacts ($=10$)",
  Dangerous_Workflow_score  = "Dangerous-Workflow ($=10$)",
  Contrib_score             = "Contributors ($=10$)"
)

# ---- 2. Formatting helpers --------------------------------------------------
stars_for <- function(p) {
  if (is.na(p)) return("")
  if (p < 0.001) return("***")
  if (p < 0.01)  return("**")
  if (p < 0.05)  return("*")
  ""
}

# Round to `digits` decimals; if that rounds a nonzero value to 0, add one
# more decimal place (repeat up to max_digits). Mirrors the Maintained/
# Code-Review 3-dp SE pattern in Table III while keeping most rows at 2dp.
fmt_num <- function(x, digits = 2, max_digits = 4) {
  d <- digits
  repeat {
    out <- formatC(x, format = "f", digits = d)
    zero_str <- formatC(0, format = "f", digits = d)
    if (out != zero_str || x == 0 || d >= max_digits) return(out)
    d <- d + 1
  }
}

# ---- 3. Pull stats out of each univariate model -----------------------------
extract_row <- function(res) {
  m  <- res$model
  ct <- coeftable(m)          # Estimate, Std. Error, z value, Pr(>|z|)
  coef_name <- rownames(ct)[1]   # first row = the check predictor
  beta <- ct[1, "Estimate"]
  se   <- ct[1, "Std. Error"]
  p    <- ct[1, "Pr(>|z|)"]
  pct  <- (exp(beta) - 1) * 100

  list(
    beta_str = paste0(fmt_num(beta, 2), stars_for(p)),
    se_str   = paste0("(", fmt_num(se, 2), ")"),
    pct_str  = paste0(ifelse(pct >= 0, "", "-"), formatC(abs(pct), format = "f", digits = 1), "\\%"),
    n_obs    = res$n_obs
  )
}

# ---- 4. Build the table body -------------------------------------------------
build_table3 <- function(results_list, row_labels) {
  rows <- character(length(row_labels))
  for (i in seq_along(row_labels)) {
    check_name <- names(row_labels)[i]
    res <- results_list[[check_name]]
    r <- extract_row(res)
    rows[i] <- sprintf(
      "%s & %s (%s) & %s \\\\",
      row_labels[i],
      r$beta_str,
      gsub("^\\(|\\)$", "", r$se_str),
      r$pct_str
    )
  }
  rows
}

body_rows <- build_table3(vuln_results, row_labels)

# ---- 5. Assemble full LaTeX table (IEEEtran / booktabs / threeparttable) ----
latex_table <- c(
  "\\begin{table}[t]",
  "\\centering",
  "\\caption{Individual Check Models, PPML with Two-Way Fixed Effects}",
  "\\label{tab:univariate_vuln}",
  "\\begin{threeparttable}",
  "\\begin{tabular}{lcc}",
  "\\toprule",
  " & \\multicolumn{2}{c}{Vulnerability Count} \\\\",
  "\\cmidrule(lr){2-3}",
  " & Log scale $\\beta$ (SE) & \\% change $\\exp(\\beta)-1$ \\\\",
  "\\midrule",
  body_rows,
  "\\midrule",
  "Package FE & \\multicolumn{2}{c}{\\checkmark} \\\\",
  "Year-Month FE & \\multicolumn{2}{c}{\\checkmark} \\\\",
  "\\bottomrule",
  "\\end{tabular}",
  "\\begin{tablenotes}",
  "\\footnotesize",
  "\\item $^*p<0.05$; $^{**}p<0.01$; $^{***}p<0.001$",
  "\\item Maintained and Code-Review are continuous. Remaining checks are binary indicators.",
  "\\item Covariates include log dependents, age, log downloads, and log release count.",
  "\\end{tablenotes}",
  "\\end{threeparttable}",
  "\\end{table}"
)

cat(paste(latex_table, collapse = "\n"), "\n")

\begin{table}[t]
\centering
\caption{Individual Check Models, PPML with Two-Way Fixed Effects}
\label{tab:univariate_vuln}
\begin{threeparttable}
\begin{tabular}{lcc}
\toprule
 & \multicolumn{2}{c}{Vulnerability Count} \\
\cmidrule(lr){2-3}
 & Log scale $\beta$ (SE) & \% change $\exp(\beta)-1$ \\
\midrule
Maintained & -0.01*** (0.003) & -1.4\% \\
Code-Review & 0.01* (0.005) & 1.0\% \\
Pinned-Dependencies ($>0$) & -0.08 (0.08) & -7.6\% \\
Security-Policy ($>0$) & 0.08 (0.08) & 8.3\% \\
Token-Permissions ($>0$) & 0.04 (0.14) & 3.6\% \\
Dependency-Update-Tool ($>0$) & 0.04 (0.06) & 4.1\% \\
Binary-Artifacts ($=10$) & -0.29*** (0.09) & -25.3\% \\
Dangerous-Workflow ($=10$) & -0.04 (0.05) & -4.0\% \\
Contributors ($=10$) & 0.13 (0.08) & 13.7\% \\
\midrule
Package FE & \multicolumn{2}{c}{\checkmark} \\
Year-Month FE & \multicolumn{2}{c}{\checkmark} \\
\bottomrule
\end{tabular}
\begin{tablenotes}
\footnotesize
\item $^*p<0.05$; $^{**}p<0.01$; $^{***}p<0.001$
\item Maintained and Code-Review a

In [3]:
# ==================== MTTU MODELS ====================
library(fixest)
library(dplyr)
library(tidyr)
library(lubridate)

# ---- 1. Dichotomization functions -------------------------
bin_zero_pos <- function(x) {
  factor(if_else(x < 0.5, "zero", "positive"),
         levels = c("zero", "positive"))
}
bin_ten_less <- function(x) {
  factor(if_else(x > 9.5, "ten", "below_ten"),
         levels = c("below_ten", "ten"))
}
binnings <- list(zero_pos = bin_zero_pos, ten_less = bin_ten_less)

chosen_binning <- c(
  Maintained_score           = "continuous",
  Code_Review_score          = "continuous",
  Pinned_Dependencies_score  = "zero_pos",
  Security_Policy_score      = "zero_pos",
  Token_Permissions_score    = "zero_pos",
  DependUpTool_score         = "zero_pos",
  Binary_Artifacts_score     = "ten_less",
  Dangerous_Workflow_score   = "ten_less",
  Contrib_score              = "ten_less"
)

controls <- "log1p(downloads_7_day_total) + log1p(dependents) + log1p(loc) + version_age_days + log1p(Release_count)"
fe_part  <- "package_name + year_month"

# ---- 2. Function to run one univariate model -----------------------------
run_univariate_model <- function(panel, check_name, binning_type, outcome_var) {
  panel_check <- panel
  if (binning_type == "continuous") {
    predictor_term <- check_name
  } else {
    bin_col <- paste0(check_name, "_bin")
    f <- binnings[[binning_type]]
    panel_check[[bin_col]] <- f(panel_check[[check_name]])
    predictor_term <- bin_col
  }
  fml_str <- paste(
    outcome_var, "~", predictor_term, "+", controls,
    "|", fe_part
  )
  model <- feglm(as.formula(fml_str), data = panel_check,
                  family = "poisson", cluster = ~package_name)
  list(model = model, n_obs = nobs(model), check = check_name, outcome = outcome_var)
}

# ---- 3. Run for every check ------------------------------------------------
panel_mttu <- read.csv("../data/monthly_panel_mttu_mttr.csv")  # update path as needed

mttu_results <- lapply(names(chosen_binning), function(check_name) {
  run_univariate_model(panel_mttu, check_name, chosen_binning[[check_name]], "MTTU")
})
names(mttu_results) <- names(chosen_binning)

# ---- 4. Print summary() for each model -------------------------------------
cat("=== MTTU: Univariate Model Summaries ===\n\n")
for (check_name in names(mttu_results)) {
  cat("----", check_name, "----\n")
  print(summary(mttu_results[[check_name]]$model))
  cat("\n")
}

# ---- 5. N and % change summary per check -----------------------------------
summarize_univariate <- function(results_list, outcome_label) {
  for (check_name in names(results_list)) {
    res <- results_list[[check_name]]
    m <- res$model
    coef_name <- names(coef(m))[1]
    pct_change <- (exp(coef(m)[coef_name]) - 1) * 100
    cat(sprintf("[%s] %s (N=%d): coef on %s -> %.2f%% change\n",
                outcome_label, check_name, res$n_obs, coef_name, pct_change))
  }
}
summarize_univariate(mttu_results, "MTTU")

NOTES: 1,961 observations removed because of NA values (RHS: 1,961).
       5,265/0 fixed-effects (15,905 observations) removed because of only 0 outcomes or singletons.

NOTES: 1,990 observations removed because of NA values (RHS: 1,990).
       5,274/0 fixed-effects (15,912 observations) removed because of only 0 outcomes or singletons.

NOTES: 9,503 observations removed because of NA values (RHS: 9,503).
       4,479/0 fixed-effects (13,815 observations) removed because of only 0 outcomes or singletons.

NOTES: 1,837 observations removed because of NA values (RHS: 1,837).
       5,272/0 fixed-effects (15,926 observations) removed because of only 0 outcomes or singletons.

NOTES: 9,451 observations removed because of NA values (RHS: 9,451).
       4,496/0 fixed-effects (13,903 observations) removed because of only 0 outcomes or singletons.

NOTES: 1,880 observations removed because of NA values (RHS: 1,880).
       5,274/0 fixed-effects (15,925 observations) removed because of only 0

=== MTTU: Univariate Model Summaries ===

---- Maintained_score ----
GLM estimation, family = poisson, Dep. Var.: MTTU
Observations: 61,197
Fixed-effects: package_name: 8,266,  year_month: 25
Standard-errors: Clustered (package_name) 
                              Estimate Std. Error   z value   Pr(>|z|)    
Maintained_score             -0.037754   0.003133 -12.04928  < 2.2e-16 ***
log1p(downloads_7_day_total) -0.007242   0.003493  -2.07350 3.8126e-02 *  
log1p(dependents)             0.020810   0.009604   2.16683 3.0247e-02 *  
log1p(loc)                   -0.066844   0.034680  -1.92745 5.3924e-02 .  
version_age_days              0.003318   0.000582   5.70527 1.1616e-08 ***
log1p(Release_count)         -0.120447   0.013436  -8.96449  < 2.2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -151,778.5   Adj. Pseudo R2: 0.707102
           BIC:  394,994.3     Squared Cor.: 0.860734

---- Code_Review_score ----
GLM estimation, family = poisson, De

In [12]:
# ==================== TABLE III: INDIVIDUAL CHECK MODELS (LaTeX) ====================
library(fixest)

# ---- 1. Row labels + ordering (matches chosen_binning order) --------------
row_labels <- c(
  Maintained_score          = "Maintained",
  Code_Review_score         = "Code-Review",
  Pinned_Dependencies_score = "Pinned-Dependencies ($>0$)",
  Security_Policy_score     = "Security-Policy ($>0$)",
  Token_Permissions_score   = "Token-Permissions ($>0$)",
  DependUpTool_score        = "Dependency-Update-Tool ($>0$)",
  Binary_Artifacts_score    = "Binary-Artifacts ($=10$)",
  Dangerous_Workflow_score  = "Dangerous-Workflow ($=10$)",
  Contrib_score             = "Contributors ($=10$)"
)

# ---- 2. Formatting helpers --------------------------------------------------
stars_for <- function(p) {
  if (is.na(p)) return("")
  if (p < 0.001) return("***")
  if (p < 0.01)  return("**")
  if (p < 0.05)  return("*")
  ""
}

# Round to `digits` decimals; if that rounds a nonzero value to 0, add one
# more decimal place (repeat up to max_digits). Mirrors the Maintained/
# Code-Review 3-dp SE pattern in Table III while keeping most rows at 2dp.
fmt_num <- function(x, digits = 2, max_digits = 4) {
  d <- digits
  repeat {
    out <- formatC(x, format = "f", digits = d)
    zero_str <- formatC(0, format = "f", digits = d)
    if (out != zero_str || x == 0 || d >= max_digits) return(out)
    d <- d + 1
  }
}

# ---- 3. Pull stats out of each univariate model -----------------------------
extract_row <- function(res) {
  m  <- res$model
  ct <- coeftable(m)          # Estimate, Std. Error, z value, Pr(>|z|)
  coef_name <- rownames(ct)[1]   # first row = the check predictor
  beta <- ct[1, "Estimate"]
  se   <- ct[1, "Std. Error"]
  p    <- ct[1, "Pr(>|z|)"]
  pct  <- (exp(beta) - 1) * 100

  list(
    beta_str = paste0(fmt_num(beta, 2), stars_for(p)),
    se_str   = paste0("(", fmt_num(se, 2), ")"),
    pct_str  = paste0(ifelse(pct >= 0, "", "-"), formatC(abs(pct), format = "f", digits = 1), "\\%"),
    n_obs    = res$n_obs
  )
}

# ---- 4. Build the table body -------------------------------------------------
build_table3 <- function(results_list, row_labels) {
  rows <- character(length(row_labels))
  for (i in seq_along(row_labels)) {
    check_name <- names(row_labels)[i]
    res <- results_list[[check_name]]
    r <- extract_row(res)
    rows[i] <- sprintf(
      "%s & %s (%s) & %s \\\\",
      row_labels[i],
      r$beta_str,
      gsub("^\\(|\\)$", "", r$se_str),
      r$pct_str
    )
  }
  rows
}

body_rows <- build_table3(mttu_results, row_labels)

# ---- 5. Assemble full LaTeX table (IEEEtran / booktabs / threeparttable) ----
latex_table <- c(
  "\\begin{table}[t]",
  "\\centering",
  "\\caption{Individual Check Models, PPML with Two-Way Fixed Effects}",
  "\\label{tab:univariate_vuln}",
  "\\begin{threeparttable}",
  "\\begin{tabular}{lcc}",
  "\\toprule",
  " & \\multicolumn{2}{c}{Vulnerability Count} \\\\",
  "\\cmidrule(lr){2-3}",
  " & Log scale $\\beta$ (SE) & \\% change $\\exp(\\beta)-1$ \\\\",
  "\\midrule",
  body_rows,
  "\\midrule",
  "Package FE & \\multicolumn{2}{c}{\\checkmark} \\\\",
  "Year-Month FE & \\multicolumn{2}{c}{\\checkmark} \\\\",
  "\\bottomrule",
  "\\end{tabular}",
  "\\begin{tablenotes}",
  "\\footnotesize",
  "\\item $^*p<0.05$; $^{**}p<0.01$; $^{***}p<0.001$",
  "\\item Maintained and Code-Review are continuous. Remaining checks are binary indicators.",
  "\\item Covariates include log dependents, age, log downloads, and log release count.",
  "\\end{tablenotes}",
  "\\end{threeparttable}",
  "\\end{table}"
)

cat(paste(latex_table, collapse = "\n"), "\n")

\begin{table}[t]
\centering
\caption{Individual Check Models, PPML with Two-Way Fixed Effects}
\label{tab:univariate_vuln}
\begin{threeparttable}
\begin{tabular}{lcc}
\toprule
 & \multicolumn{2}{c}{Vulnerability Count} \\
\cmidrule(lr){2-3}
 & Log scale $\beta$ (SE) & \% change $\exp(\beta)-1$ \\
\midrule
Maintained & -0.04*** (0.003) & -3.7\% \\
Code-Review & 0.003 (0.004) & 0.3\% \\
Pinned-Dependencies ($>0$) & -0.05 (0.05) & -4.7\% \\
Security-Policy ($>0$) & -0.17** (0.06) & -15.8\% \\
Token-Permissions ($>0$) & -0.01 (0.05) & -1.3\% \\
Dependency-Update-Tool ($>0$) & -0.03 (0.05) & -3.0\% \\
Binary-Artifacts ($=10$) & -0.14* (0.07) & -13.0\% \\
Dangerous-Workflow ($=10$) & 0.04 (0.06) & 4.2\% \\
Contributors ($=10$) & 0.001 (0.07) & 0.1\% \\
\midrule
Package FE & \multicolumn{2}{c}{\checkmark} \\
Year-Month FE & \multicolumn{2}{c}{\checkmark} \\
\bottomrule
\end{tabular}
\begin{tablenotes}
\footnotesize
\item $^*p<0.05$; $^{**}p<0.01$; $^{***}p<0.001$
\item Maintained and Code-Rev